# Train YOLO26 — Drowsy Driver Detection
**Trước khi chạy:** `Runtime` → `Change runtime type` → **T4 GPU**

In [ ]:
# Cell 1 — Kiểm tra GPU
!nvidia-smi

In [ ]:
# Cell 2 — Cài ultralytics + supervision
%pip install -q "ultralytics>=8.4.0" supervision
!yolo settings sync=False
import ultralytics
ultralytics.checks()

In [ ]:
# Cell 3 — Thiết lập HOME (bắt buộc chạy trước)
import os
HOME = os.getcwd()
print('HOME:', HOME)

In [ ]:
# Cell 4 — Download dataset từ Roboflow
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="qI3lEKlNpIZpNENdk3MH")
project = rf.workspace("nguyen-tuan-dat").project("datio_drowsines")
dataset = project.version(1).download("yolo26")

In [ ]:
# Cell 5 — Xem thông tin dataset
%cd {HOME}
import yaml
from pathlib import Path

yaml_path = Path(dataset.location) / 'data.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

print('Classes:', cfg['names'])
print('Num classes:', cfg['nc'])
for split in ['train', 'valid', 'val', 'test']:
    p = Path(dataset.location) / split / 'images'
    if p.exists():
        n = len(list(p.glob('*.jpg')) + list(p.glob('*.png')))
        print(f'  {split}: {n} images')

In [ ]:
# Cell 6 — Xem mẫu ảnh
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

train_dir = None
for name in ['train', 'valid']:
    p = Path(dataset.location) / name / 'images'
    if p.exists():
        train_dir = p
        break

if train_dir:
    imgs = list(train_dir.glob('*.jpg'))[:6]
    imgs += list(train_dir.glob('*.png'))[:max(0, 6-len(imgs))]
    imgs = imgs[:6]
    if imgs:
        fig, axes = plt.subplots(2, 3, figsize=(12, 7))
        for ax, p in zip(axes.flat, imgs):
            ax.imshow(mpimg.imread(str(p)))
            ax.axis('off')
        for ax in axes.flat[len(imgs):]:
            ax.axis('off')
        plt.suptitle('Mau dataset', fontsize=13)
        plt.tight_layout()
        plt.show()
else:
    print('Khong tim thay thu muc anh')

In [ ]:
# Cell 7 — TRAIN YOLO26
!yolo task=detect \
      mode=train \
      model=yolo26m.pt \
      data={dataset.location}/data.yaml \
      epochs=50 \
      imgsz=640 \
      batch=16 \
      patience=15 \
      plots=True \
      name=drowsy_yolo26 \
      project={HOME}/runs/detect

In [ ]:
# Cell 8 — Xem kết quả
from IPython.display import Image as IPyImage
results_path = f'{HOME}/runs/detect/drowsy_yolo26'
!ls {results_path}
IPyImage(filename=f'{results_path}/results.png', width=900)

In [ ]:
# Cell 9 — Confusion matrix
from IPython.display import Image as IPyImage
IPyImage(filename=f'{HOME}/runs/detect/drowsy_yolo26/confusion_matrix.png', width=600)

In [ ]:
# Cell 10 — Validate
best = f'{HOME}/runs/detect/drowsy_yolo26/weights/best.pt'
!yolo task=detect mode=val model={best} data={dataset.location}/data.yaml

In [ ]:
# Cell 11 — Predict + hiển thị
best = f'{HOME}/runs/detect/drowsy_yolo26/weights/best.pt'
!yolo task=detect mode=predict model={best} \
      source={dataset.location}/test/images conf=0.3 save=True verbose=False

import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg
pred_imgs = sorted(glob.glob(f'{HOME}/runs/detect/predict*/*.jpg'))[:4]
if pred_imgs:
    fig, axes = plt.subplots(1, len(pred_imgs), figsize=(16, 5))
    if len(pred_imgs) == 1: axes = [axes]
    for ax, p in zip(axes, pred_imgs):
        ax.imshow(mpimg.imread(p)); ax.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 12 — Lưu summary JSON
import pandas as pd, json
df = pd.read_csv(f'{HOME}/runs/detect/drowsy_yolo26/results.csv')
df.columns = [c.strip() for c in df.columns]
map50_col = [c for c in df.columns if 'map50' in c.lower() and '95' not in c.lower()]
best_idx = df[map50_col[0]].idxmax()
best_map50 = float(df[map50_col[0]].iloc[best_idx])
best_epoch = int(df['epoch'].iloc[best_idx])
print(f'Best mAP@50: {best_map50*100:.2f}%  @ epoch {best_epoch}')

summary = {'model':'yolo26m','device':'colab_t4',
           'epochs_ran':len(df),'best_epoch':best_epoch,
           'best_map50':round(best_map50,6),'classes':cfg.get('names',[])}
with open(f'{HOME}/summary_yolo26.json','w') as f:
    json.dump(summary, f, indent=2)
print('Saved: summary_yolo26.json')

In [ ]:
# Cell 13 — Download về máy
from google.colab import files
files.download(f'{HOME}/summary_yolo26.json')
files.download(f'{HOME}/runs/detect/drowsy_yolo26/weights/best.pt')